# Introduction to Agentic AI — Direct vs. ReAct

> **API Key：优先读取环境变量；未设置时使用 `getpass()` 临时输入。**

本Notebook用同一个智谱模型完成同一道两阶段数学题。Direct只有一次生成；ReAct可以调用计算器、读取Observation，再决定下一步Action。

**学习目标**

- 看懂 `Thought → Action → Observation` 循环；
- 体验前一步Observation如何改变下一步Action；
- 理解工具接口、JSON解析、验证器和步数上限为什么属于Harness。


## 1. 挑战题

门禁系统按以下规则生成六位验证码：

1. 计算 $S=20250807^{123457}\bmod 1000000$；
2. 观察 $S$：若为偶数，$C=(S\times2026+314159)\bmod1000000$；若为奇数，$C=(S\times2025+271828)\bmod1000000$；
3. 输出 ` {"answer":"......"} `。

两种模式看到完全相同的题目。Direct没有计算器和第二次机会；ReAct最多调用两次计算器，然后Finish。


In [ ]:
import ast
import json
import operator
import os
import re
import urllib.error
import urllib.request
from getpass import getpass

MODEL = os.getenv("ZAI_MODEL", "glm-4-flash-250414")
USE_REAL_API = True       # True=真实API；False=免费离线演示
MAX_STEPS = 6

TASK = '''Compute the six-digit access code.
1. S = 20250807^123457 mod 1000000.
2. If S is even: C = (S * 2026 + 314159) mod 1000000.
   If S is odd:  C = (S * 2025 + 271828) mod 1000000.
3. Return exactly {"answer":"......"}. Keep leading zeroes.'''
EXPECTED = "156003"
REQUIRED_OBSERVATIONS = {"730807", "156003"}
print("Model:", MODEL, "| Real API:", USE_REAL_API)


## 2. 配置API Key

### 推荐方式：让VS Code从终端继承环境变量

Linux / macOS先完全关闭VS Code，然后执行：

```bash
export ZAI_API_KEY="你的API Key"
code /home/yiyunzhou/course/course_code/01Introduction
```

Windows PowerShell使用 `$env:ZAI_API_KEY="你的API Key"`，再从同一个PowerShell执行 `code .`。Notebook通过 `os.getenv("ZAI_API_KEY")` 读取。

### 课堂便捷方式：`getpass()`临时输入

如果没有检测到环境变量，运行模型客户端单元格时会调用 `getpass()`。输入内容不会回显，也不会写入Notebook输出。

该方式仅用于课堂临时操作；Key会保留在当前Kernel内存中，因此课后应 **Restart Kernel**，并轮换课堂共享Key。

> 不要把真实Key写入 `%env`、`os.environ[...]` 或普通Python字符串，因为它们可能随Notebook保存。


In [ ]:
class ZhipuClient:
    endpoint = "https://open.bigmodel.cn/api/paas/v4/chat/completions"

    def __init__(self, api_key=None):
        self.api_key = api_key or os.getenv("ZAI_API_KEY")
        if not self.api_key:
            print("未检测到环境变量ZAI_API_KEY，请临时输入课堂API Key。")
            self.api_key = getpass("Zhipu API Key（输入内容不会显示）: " )
        if not self.api_key:
            raise ValueError("Missing API key")

    def chat(self, messages, temperature=0.2, max_tokens=500):
        body = json.dumps({"model": MODEL, "messages": messages,
                           "temperature": temperature, "max_tokens": max_tokens}).encode("utf-8")
        request = urllib.request.Request(
            self.endpoint, data=body, method="POST",
            headers={"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"Zhipu API error {exc.code}: {detail}") from exc
        return payload["choices"][0]["message"]["content"]


class OfflineClient:
    def __init__(self):
        pass

    def chat(self, messages, temperature=0.2, max_tokens=500):
        if "DIRECT_BASELINE" in messages[0]["content"]:
            return '{"answer":"482731"}'
        observations = [m["content"] for m in messages
                        if m["role"] == "user" and m["content"].startswith("Observation:")]
        if not observations:
            return "Thought: I need the seed before choosing a branch.\nAction: Calculate[20250807^123457 % 1000000]"
        if len(observations) == 1 and "730807" in observations[-1]:
            return "Thought: 730807 is odd, so use the odd branch.\nAction: Calculate[(730807*2025+271828)%1000000]"
        if len(observations) >= 2 and "156003" in observations[-1]:
            return 'Thought: The final code is verified.\nAction: Finish[{"answer":"156003"}]'
        return "Thought: I still need a usable Observation.\nAction: Calculate[20250807^123457 % 1000000]"

client = ZhipuClient() if USE_REAL_API else OfflineClient()


## 3. Direct：一次生成、没有工具

先预测：弱模型能够在一次生成中稳定完成大指数模运算吗？


In [ ]:
DIRECT_SYSTEM = '''DIRECT_BASELINE
Answer once. You have no calculator, code execution, tools, or second attempt.
Return the requested JSON and do not claim to have used a tool.'''.strip()

direct_text = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                           {"role": "user", "content": TASK}])
print(direct_text)


## 4. 唯一的外部工具：安全计算器

计算器只接受数字算术和三参数 `pow(base, exponent, modulus)`。考虑到弱模型常输出数学记号，它还会把严格的纯数字 `a^b % m` 安全转换为 `pow(a,b,m)`，但不会执行任意Python代码。


In [ ]:
BIN_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod}

def safe_calculate(expression):
    expression = expression.strip().strip('`').replace('×', '*').replace('÷', '/')
    expression = re.sub(r'\bmod\b', '%', expression, flags=re.I)
    caret = re.fullmatch(r'\s*([+-]?\d+)\s*\^\s*(\d+)\s*%\s*(\d+)\s*', expression)
    if caret: expression = 'pow({},{},{})'.format(*caret.groups())

    def visit(node):
        if isinstance(node, ast.Expression): return visit(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in BIN_OPS:
            value = BIN_OPS[type(node.op)](visit(node.left), visit(node.right))
            if abs(value) > 10**15: raise ValueError('Intermediate result is too large')
            return value
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == 'pow':
            if node.keywords or len(node.args) != 3: raise ValueError('Use pow(base, exponent, modulus)')
            base, exponent, modulus = [visit(arg) for arg in node.args]
            if not all(isinstance(x, int) for x in (base, exponent, modulus)):
                raise ValueError('pow arguments must be integers')
            if not (0 <= exponent <= 10**9 and 0 < modulus <= 10**12 and abs(base) <= 10**12):
                raise ValueError('pow arguments are outside the safe range')
            return pow(base, exponent, modulus)
        raise ValueError('Only numeric arithmetic is allowed')

    value = visit(ast.parse(expression, mode='eval'))
    return str(int(value) if isinstance(value, float) and value.is_integer() else value)

assert safe_calculate('20250807^123457 % 1000000') == '730807'
assert safe_calculate('(730807*2025+271828)%1000000') == '156003'
print('Calculator checks passed.')


## 5. 课堂任务：补全ReAct的三个关键连接（约12–15分钟）

解析Action、JSON容错、循环和步数上限已经提供，学生不需要从零实现Agent。请完成下面三个短TODO，每个只需一行：

1. **Action → Tool**：执行Calculate动作；
2. **Tool result → Observation**：把工具结果放回messages；
3. **Finish / Stop**：只有最终答案和两次关键计算都正确时才允许结束。

完成后运行后续单元格，检查轨迹是否形成 `Action → Observation → 下一次Action → Finish`。


In [ ]:
def execute_calculate(action):
    # TODO 1：action[1]是模型给出的数学表达式；调用上方安全计算器并返回结果。
    # 提示：使用 safe_calculate(...)。
    pass

def append_observation(messages, observation):
    # TODO 2：把工具结果作为下一轮模型可见的Observation加入messages。
    # 提示：role使用user；content以 "Observation:" 开头。
    pass

def finish_is_valid(answer, calculator_outputs):
    # TODO 3：返回一个布尔值。答案必须等于EXPECTED，且两次关键结果都已出现。
    # 提示：使用 REQUIRED_OBSERVATIONS <= set(calculator_outputs)。
    pass


## 6. 已提供的ReAct框架（直接运行，无需学生修改）

下面单元格默认折叠。它负责Action解析、JSON容错、循环和最大步数；课堂重点是观察你补全的工具调用、Observation回填和停止条件如何共同构成ReAct闭环。


In [ ]:
ACTION_RE = re.compile(r'^\s*Action\s*:\s*(Calculate|Finish)\s*\[(.*?)\]\s*$', re.I | re.M | re.S)

def extract_answer(text):
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try: value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError: continue
        if isinstance(value, dict) and 'answer' in value: return str(value['answer']).strip()
    for candidate in re.findall(r'\{[^{}]{1,300}\}', text, re.S):
        try: value = ast.literal_eval(candidate)
        except (SyntaxError, ValueError): continue
        if isinstance(value, dict) and 'answer' in value: return str(value['answer']).strip()
    fallback = re.search(r"[\"']?answer[\"']?\s*[:=]\s*[\"']?(\d+)", text, re.I)
    return fallback.group(1) if fallback else None

REACT_SYSTEM = '''Use a concise ReAct loop. Output exactly one action per turn:
Action: Calculate[numeric expression]
Action: Finish[{"answer":"......"}]
First calculate S. Observe its parity. Then calculate only the matching branch.
Use pow(base, exponent, 1000000) or base^exponent % 1000000. Never guess.'''.strip()

def run_react(client, task, max_steps=6):
    messages = [{"role": "system", "content": REACT_SYSTEM},
                {"role": "user", "content": task}]
    trace, calculator_outputs = [], []
    for step in range(1, max_steps + 1):
        model_text = client.chat(messages)
        messages.append({"role": "assistant", "content": model_text})
        match = ACTION_RE.search(model_text)
        action = (match.group(1).title(), match.group(2).strip()) if match else None
        if action is None:
            bare_answer = extract_answer(model_text)
            action = ('Finish', model_text) if bare_answer else None
        if action is None:
            observation = 'Format error: use exactly one Calculate[...] or Finish[{...}].'
        elif action[0] == 'Calculate':
            try: observation = execute_calculate(action)
            except (SyntaxError, ValueError, ZeroDivisionError, OverflowError) as exc:
                observation = f'Calculation error: {exc}'
            calculator_outputs.append(observation)
        else:
            answer = extract_answer(action[1]) or extract_answer(model_text)
            if finish_is_valid(answer, calculator_outputs):
                trace.append({"step": step, "model": model_text, "action": action, "observation": None})
                return {"answer": answer, "trace": trace, "passed": True, "reason": "finish"}
            observation = 'Finish blocked: calculate S, select its branch, and submit the verified six-digit result.'
        trace.append({"step": step, "model": model_text, "action": action, "observation": observation})
        append_observation(messages, observation)
    return {"answer": None, "trace": trace, "passed": False, "reason": "max_steps"}


In [ ]:
react_result = run_react(client, TASK, MAX_STEPS)
for item in react_result['trace']:
    print(f"\n--- Step {item['step']} ---")
    print(item['model'])
    if item['observation'] is not None: print('Observation:', item['observation'])
print('\nResult:', react_result)


## 7. 运行与验收（约3分钟）

补全TODO后，从ReAct框架单元格开始向下运行。作业通过必须同时满足：

- 三个TODO均已补全，不再包含 `pass`；
- 轨迹包含3步：第一次Calculate、第二次Calculate、最后Finish；
- 第一个Observation为 `730807`；
- 第二个Observation为 `156003`；
- 最终显示 `ReAct: answer='156003', pass=True`。

**提交物：** 保存后的 `Introduction_ReAct_Lab.ipynb`。提交前确认Notebook中没有API Key。

<details><summary>教师演示或课后检查：三个TODO参考答案</summary>

```python
def execute_calculate(action):
    return safe_calculate(action[1])

def append_observation(messages, observation):
    messages.append({
        "role": "user",
        "content": f"Observation: {observation}\nContinue with exactly one Action.",
    })

def finish_is_valid(answer, calculator_outputs):
    return answer == EXPECTED and REQUIRED_OBSERVATIONS <= set(calculator_outputs)
```

</details>


In [ ]:
direct_answer = extract_answer(direct_text)
print(f"Direct: answer={direct_answer!r}, pass={direct_answer == EXPECTED}")
print(f"ReAct: answer={react_result['answer']!r}, pass={react_result['passed']}")
if react_result['passed']:
    print('完成：Action、Observation与Finish已形成完整ReAct闭环。')
else:
    print('尚未完成：请检查Tool、Observation和Finish三个TODO。')
